In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import yaml
import time

# Load gene info

In [2]:
fpath = "../../resources/isoform_map.csv.gz"
gdf = pd.read_csv(fpath, low_memory=False)

# assume your DataFrame is named df
# 1. Map Gene stable ID → Gene name
gene_map = (
    gdf[['Gene stable ID', 'Gene name']]
    .drop_duplicates()
    .set_index('Gene stable ID')['Gene name']
    .to_dict()
)

# 2. Map Transcript stable ID → Transcript name
transcript_map = (
    gdf[['Transcript stable ID', 'Transcript name']]
    .drop_duplicates()
    .set_index('Transcript stable ID')['Transcript name']
    .to_dict()
)

print(f"{len(gene_map)=}")
print(f"{len(transcript_map)=}")

# # Example usage
print(gene_map.get('ENSG00000150907'))
print(transcript_map.get('ENST00000361390'))

gdf.head()

len(gene_map)=21517
len(transcript_map)=109780
FOXO1
MT-ND1-201


,Gene stable ID,Gene stable ID version,Transcript stable ID,Transcript stable ID version,Protein stable ID,Protein stable ID version,Transcript length (including UTRs and CDS),Transcript name,Gene name,UniProtKB/Swiss-Prot ID,UniProtKB/TrEMBL ID
0,ENSG00000198888,ENSG00000198888.2,ENST00000361390,ENST00000361390.2,ENSP00000354687,ENSP00000354687.2,956,MT-ND1-201,MT-ND1,P03886,U5Z754
1,ENSG00000198763,ENSG00000198763.3,ENST00000361453,ENST00000361453.3,ENSP00000355046,ENSP00000355046.4,1042,MT-ND2-201,MT-ND2,P03891,Q7GXY9
2,ENSG00000198804,ENSG00000198804.2,ENST00000361624,ENST00000361624.2,ENSP00000354499,ENSP00000354499.2,1542,MT-CO1-201,MT-CO1,P00395,U5YWV7
3,ENSG00000198712,ENSG00000198712.1,ENST00000361739,ENST00000361739.1,ENSP00000354876,ENSP00000354876.1,684,MT-CO2-201,MT-CO2,P00403,U5Z487
4,ENSG00000228253,ENSG00000228253.1,ENST00000361851,ENST00000361851.1,ENSP00000355265,ENSP00000355265.1,207,MT-ATP8-201,MT-ATP8,P03928,U5YV54


# Load SUPPA outputs

In [3]:
dirpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/"

file_list = glob.glob(f"{dirpath}/*.ioe*")
file_list

['/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_AF_strict.ioe',
 '/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_MX_strict.ioe',
 '/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_AL_strict.ioe',
 '/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_RI_strict.ioe',
 '/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_A5_strict.ioe',
 '/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_A3_strict.ioe',
 '/nfs/turbo/umms-indikar/shared/projects/HSC/data/resources/suppa_AS_events/as_events_SE_strict.ioe']

In [4]:
# from https://github.com/comprna/SUPPA?tab=readme-ov-file#ioe
event_descriptions = {
    "SE": "skipping exon",
    "A5": "alternative 5' splice site",
    "A3": "alternative 3' splice site",
    "MX": "mutually exclusive exon",
    "RI": "retained intron",
    "AF": "alternative first exon",
    "AL": "alternative last exon",
}

In [5]:
df_list = []

for file_path in file_list:
    basename = os.path.basename(file_path)

    event_type = basename.split("_")[2]
    tmp = pd.read_csv(file_path, sep='\t')
    tmp = tmp[['gene_id', 'alternative_transcripts']]
    print(f"raw {tmp.shape=}")

    # 1. split comma-separated strings into lists
    tmp['alternative_transcripts'] = (
        tmp['alternative_transcripts']
        .astype(str)
        .str.split(',')
    )
    
    # 2. explode lists into one row per transcript
    tmp = (
        tmp
        .explode('alternative_transcripts')
        .reset_index(drop=True)
    )
    
    # 3. strip whitespace
    tmp['alternative_transcripts'] = tmp['alternative_transcripts'].str.strip()
    tmp = tmp.drop_duplicates()
    print(f"exploded {tmp.shape=}")
    tmp['event_type'] = event_type
    tmp['event_label'] = event_descriptions[event_type]
    tmp['gene_name'] = tmp['gene_id'].map(gene_map)
    tmp['transcript_name'] = tmp['alternative_transcripts'].map(transcript_map)

    df_list.append(tmp)


df = pd.concat(df_list)
df = df.reset_index(drop=True)

df = df.rename(columns={'alternative_transcripts' : 'transcript_id'})

print(f"\n{df.shape=}")
print(f"{df['gene_id'].nunique()=}")
print(f"{df['transcript_id'].nunique()=}")
df.head()

raw tmp.shape=(128508, 2)
exploded tmp.shape=(49128, 2)
raw tmp.shape=(10205, 2)
exploded tmp.shape=(13512, 2)
raw tmp.shape=(44050, 2)
exploded tmp.shape=(21999, 2)
raw tmp.shape=(10398, 2)
exploded tmp.shape=(12647, 2)
raw tmp.shape=(21366, 2)
exploded tmp.shape=(41585, 2)
raw tmp.shape=(23846, 2)
exploded tmp.shape=(56299, 2)
raw tmp.shape=(55878, 2)
exploded tmp.shape=(93285, 2)

df.shape=(288455, 6)
df['gene_id'].nunique()=20135
df['transcript_id'].nunique()=149099


,gene_id,transcript_id,event_type,event_label,gene_name,transcript_name
0,ENSG00000236601,ENST00000450983,AF,alternative first exon,NaN,NaN
1,ENSG00000236601,ENST00000412666,AF,alternative first exon,NaN,NaN
2,ENSG00000237491,ENST00000655765,AF,alternative first exon,NaN,NaN
3,ENSG00000237491,ENST00000670700,AF,alternative first exon,NaN,NaN
4,ENSG00000237491,ENST00000429505,AF,alternative first exon,NaN,NaN


# Save to file

In [6]:
outpath = "../../resources/AS_events.csv.gz"
df.to_csv(
    outpath,
    index=False,
    compression='gzip',
)

df.head()

,gene_id,transcript_id,event_type,event_label,gene_name,transcript_name
0,ENSG00000236601,ENST00000450983,AF,alternative first exon,NaN,NaN
1,ENSG00000236601,ENST00000412666,AF,alternative first exon,NaN,NaN
2,ENSG00000237491,ENST00000655765,AF,alternative first exon,NaN,NaN
3,ENSG00000237491,ENST00000670700,AF,alternative first exon,NaN,NaN
4,ENSG00000237491,ENST00000429505,AF,alternative first exon,NaN,NaN
